In [1]:
import pandas as pd
import numpy as np 

from pathlib import Path 
import os 
import yaml

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split


In [2]:
os.chdir("..")
print(os.getcwd())

e:\project_archive\student-dropout-enrolled-graduate-rate-prediction


===========================  LOAD DATA  ================================

In [5]:
DATA_PATH = Path("data/processed")

X_train_path = DATA_PATH / "X_train.csv"
y_train_path = DATA_PATH / "y_train.npy"

if not X_train_path.exists():
    raise FileExistsError

if not y_train_path.exists():
    raise FileExistsError

In [11]:
X_train = pd.read_csv(X_train_path, sep=';')
y_train = np.load(y_train_path)

In [7]:
print("X_train shape: ",X_train.shape)
print("y_train shape: ",len(y_train))

X_train shape:  (3539, 1)
y_train shape:  3539


In [12]:
X_train.head()

,Marital status,Application mode,Application order,Course,Previous qualification,Previous qualification (grade),Mother's qualification,Father's qualification,Mother's occupation,Father's occupation,...,Debtor,Tuition fees up to date,Gender,Scholarship holder,Age at enrollment,Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (grade),Inflation rate
0,4,7,1,9147,3,130.0,19,1,5,5,...,0,1,0,0,35,5,5,0,0.000000,0.6
1,1,39,1,9085,1,130.0,37,37,6,6,...,0,1,0,1,25,6,13,3,11.666667,0.6
2,1,1,6,9070,6,119.0,1,1,9,9,...,0,1,1,0,22,6,6,6,14.166667,1.4
3,2,39,1,9238,19,133.1,37,37,9,4,...,0,1,1,0,42,6,0,0,0.000000,2.8
4,1,1,3,9500,1,142.0,37,38,9,9,...,0,1,0,1,22,7,7,6,13.900000,2.6


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [9]:
MODELS = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(random_state=42),
    "ExtraTrees": ExtraTreesClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42),
    "LightGBM": LGBMClassifier(random_state=42),
    "CatBoost": CatBoostClassifier(verbose=0)
}

In [14]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate

import pandas as pd
import wandb

# ---------------------------------------------------
# Cross Validation Strategy
# ---------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# ---------------------------------------------------
# Metrics
# ---------------------------------------------------

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",
    "f1_weighted": "f1_weighted",
}

results = []

# ---------------------------------------------------
# Benchmark Models
# ---------------------------------------------------

for model_name, model in MODELS.items():

    pipeline = Pipeline([
        ("classifier", model)
    ])

    scores = cross_validate(
        estimator=pipeline,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )

    metrics = {
        "model": model_name,

        "accuracy_mean": scores["test_accuracy"].mean(),
        "accuracy_std": scores["test_accuracy"].std(),

        "balanced_accuracy_mean": scores["test_balanced_accuracy"].mean(),
        "balanced_accuracy_std": scores["test_balanced_accuracy"].std(),

        "precision_macro_mean": scores["test_precision_macro"].mean(),
        "precision_macro_std": scores["test_precision_macro"].std(),

        "recall_macro_mean": scores["test_recall_macro"].mean(),
        "recall_macro_std": scores["test_recall_macro"].std(),

        "f1_macro_mean": scores["test_f1_macro"].mean(),
        "f1_macro_std": scores["test_f1_macro"].std(),

        "f1_weighted_mean": scores["test_f1_weighted"].mean(),
        "f1_weighted_std": scores["test_f1_weighted"].std(),

        "fit_time_mean": scores["fit_time"].mean(),
        "score_time_mean": scores["score_time"].mean(),
    }

    results.append(metrics)

    # -------------------------
    # Weights & Biases
    # -------------------------

    run = wandb.init(
        project="student-success-experiment-03",
        name=f"baseline-{model_name}",
        job_type="benchmark",
        group="baseline_models",
        tags=["baseline", "cross-validation"],
        config={
            "model": model_name,
            "cv_strategy": "StratifiedKFold",
            "n_splits": 5,
            "shuffle": True,
            "random_state": 42,
            "dataset": "Student Success",
            "task": "Multiclass Classification",
        },
    )

    # Log metrics
    wandb.log({
        "accuracy": metrics["accuracy_mean"],
        "balanced_accuracy": metrics["balanced_accuracy_mean"],
        "precision_macro": metrics["precision_macro_mean"],
        "recall_macro": metrics["recall_macro_mean"],
        "f1_macro": metrics["f1_macro_mean"],
        "f1_weighted": metrics["f1_weighted_mean"],
        "fit_time": metrics["fit_time_mean"],
        "score_time": metrics["score_time_mean"],
    })

    # Log standard deviations
    wandb.log({
        "accuracy_std": metrics["accuracy_std"],
        "balanced_accuracy_std": metrics["balanced_accuracy_std"],
        "precision_macro_std": metrics["precision_macro_std"],
        "recall_macro_std": metrics["recall_macro_std"],
        "f1_macro_std": metrics["f1_macro_std"],
        "f1_weighted_std": metrics["f1_weighted_std"],
    })

    wandb.summary.update(metrics)

    run.finish()

# ---------------------------------------------------
# Results
# ---------------------------------------------------

benchmark_df = (
    pd.DataFrame(results)
    .sort_values("f1_macro_mean", ascending=False)
    .reset_index(drop=True)
)

display(benchmark_df)

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


accuracy,▁
accuracy_std,▁
balanced_accuracy,▁
balanced_accuracy_std,▁
f1_macro,▁
f1_macro_std,▁
f1_weighted,▁
f1_weighted_std,▁
fit_time,▁
precision_macro,▁
precision_macro_std,▁


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


accuracy,▁
accuracy_std,▁
balanced_accuracy,▁
balanced_accuracy_std,▁
f1_macro,▁
f1_macro_std,▁
f1_weighted,▁
f1_weighted_std,▁
fit_time,▁
precision_macro,▁
precision_macro_std,▁


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


...

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


accuracy,▁
accuracy_std,▁
balanced_accuracy,▁
balanced_accuracy_std,▁
f1_macro,▁
f1_macro_std,▁
f1_weighted,▁
f1_weighted_std,▁
fit_time,▁
precision_macro,▁
precision_macro_std,▁


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


accuracy,▁
accuracy_std,▁
balanced_accuracy,▁
balanced_accuracy_std,▁
f1_macro,▁
f1_macro_std,▁
f1_weighted,▁
f1_weighted_std,▁
fit_time,▁
precision_macro,▁
precision_macro_std,▁


,model,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,f1_macro_mean,f1_macro_std,f1_weighted_mean,f1_weighted_std,fit_time_mean,score_time_mean
0,XGBoost,0.740888,0.011229,0.656681,0.013234,0.682017,0.016231,0.656681,0.013234,0.664249,0.014129,0.729843,0.011250,1.062514,0.037903
1,LightGBM,0.740888,0.009786,0.653992,0.012683,0.678920,0.013412,0.653992,0.012683,0.660491,0.014027,0.728343,0.010675,7.958842,0.030422
2,CatBoost,0.736929,0.012401,0.645824,0.012370,0.671474,0.016366,0.645824,0.012370,0.651983,0.012580,0.722989,0.011224,19.545454,0.037180
3,RandomForest,0.742017,0.004554,0.643249,0.009457,0.683493,0.016998,0.643249,0.009457,0.650204,0.012232,0.722657,0.006480,0.867792,0.033881
4,ExtraTrees,0.727891,0.003506,0.625033,0.008033,0.669896,0.013493,0.625033,0.008033,0.632333,0.009641,0.706582,0.005972,0.804806,0.053675
5,LogisticRegression,0.687767,0.004485,0.567252,0.008532,0.601755,0.018947,0.567252,0.008532,0.556696,0.015862,0.649962,0.009215,1.269004,0.025362


In [15]:
benchmark_df

,model,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,f1_macro_mean,f1_macro_std,f1_weighted_mean,f1_weighted_std,fit_time_mean,score_time_mean
0,XGBoost,0.740888,0.011229,0.656681,0.013234,0.682017,0.016231,0.656681,0.013234,0.664249,0.014129,0.729843,0.011250,1.062514,0.037903
1,LightGBM,0.740888,0.009786,0.653992,0.012683,0.678920,0.013412,0.653992,0.012683,0.660491,0.014027,0.728343,0.010675,7.958842,0.030422
2,CatBoost,0.736929,0.012401,0.645824,0.012370,0.671474,0.016366,0.645824,0.012370,0.651983,0.012580,0.722989,0.011224,19.545454,0.037180
3,RandomForest,0.742017,0.004554,0.643249,0.009457,0.683493,0.016998,0.643249,0.009457,0.650204,0.012232,0.722657,0.006480,0.867792,0.033881
4,ExtraTrees,0.727891,0.003506,0.625033,0.008033,0.669896,0.013493,0.625033,0.008033,0.632333,0.009641,0.706582,0.005972,0.804806,0.053675
5,LogisticRegression,0.687767,0.004485,0.567252,0.008532,0.601755,0.018947,0.567252,0.008532,0.556696,0.015862,0.649962,0.009215,1.269004,0.025362


In [16]:
SAVE_BENCHMARK_PATH = Path("artifacts/reports/benchmark_result.csv")

benchmark_df.to_csv(SAVE_BENCHMARK_PATH)